# Synthèse des résultats et interprétation

        Ce notebook rassemble les résultats principaux du diagnostic pré-optimisation et de l’optimisation du portefeuille d’actifs financiers de Maghrebia. Il ne relance pas les calculs lourds : il lit les exports validés des notebooks 01 et 02, reprend les tableaux clés et affiche les graphiques déjà générés.

        L’objectif est de fournir une lecture claire pour l’encadrante et pour une restitution PFE : ce qui a été observé, ce qui a été optimisé, ce que les résultats suggèrent pour Maghrebia, et quelles limites doivent rester visibles.

## 1. Objectif du notebook de synthèse

        Le notebook 01 construit les données de marché, les rendements historiques, la covariance et les scénarios APT. Le notebook 02 applique les modèles d’optimisation, les stress tests, le scoring multicritère et l’allocation additionnelle de 10 MD.

        Ce troisième notebook sert uniquement de restitution. Il ne refait ni la valorisation obligataire, ni l’estimation APT, ni l’optimisation complète. Cette séparation évite de mélanger la méthodologie avec la lecture finale des résultats.

In [1]:
from pathlib import Path
import shutil
import warnings

import numpy as np
import pandas as pd
from IPython.display import IFrame, Markdown, display

warnings.filterwarnings("ignore", category=UserWarning)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "exports").exists() and PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_EXPORTS = PROJECT_ROOT / "data" / "exports"
EXPORTS = PROJECT_ROOT / "exports"
NB01_XLSX = DATA_EXPORTS / "diagnostic_pre_optimisation_2025.xlsx"
NB02_XLSX = EXPORTS / "notebook_02_optimisation_resultats.xlsx"
FIG_NB01 = EXPORTS / "figures" / "notebook_01"
FIG_NB02 = EXPORTS / "figures" / "notebook_02"
FIG_ADD10 = FIG_NB02 / "additional_10md"
FIG_SYNTH = EXPORTS / "figures" / "notebook_03_synthese"
FIG_SYNTH.mkdir(parents=True, exist_ok=True)

SYNTH_XLSX = EXPORTS / "notebook_03_synthese_resultats.xlsx"

for required in [NB01_XLSX, NB02_XLSX]:
    if not required.exists():
        raise FileNotFoundError(f"Export requis introuvable : {required}")


def read_sheet(path: Path, sheet_name: str, fallback: pd.DataFrame | None = None) -> pd.DataFrame:
    try:
        return pd.read_excel(path, sheet_name=sheet_name)
    except Exception:
        if fallback is not None:
            return fallback.copy()
        return pd.DataFrame()


def read_csv(path: Path) -> pd.DataFrame:
    if path.exists():
        return pd.read_csv(path)
    return pd.DataFrame()


def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = [str(c).strip() for c in out.columns]
    return out


def pct_cols(df: pd.DataFrame) -> list[str]:
    keywords = ("Return", "Volatility", "VaR", "CVaR", "Gap", "Weight", "R_", "TSR")
    return [c for c in df.columns if any(k in str(c) for k in keywords)]


def display_table(df: pd.DataFrame, columns: list[str] | None = None, n: int = 12, percent: bool = True):
    if df is None or df.empty:
        display(Markdown("_Table non disponible dans les exports._"))
        return
    view = clean_columns(df)
    if columns is not None:
        existing = [c for c in columns if c in view.columns]
        if existing:
            view = view[existing]
    view = view.head(n)
    fmt = {}
    if percent:
        for col in pct_cols(view):
            if pd.api.types.is_numeric_dtype(view[col]):
                fmt[col] = "{:.2%}"
    display(view.style.format(fmt, na_rep=""))


def copy_figure(src: Path) -> Path | None:
    if not src.exists():
        return None
    dst = FIG_SYNTH / src.name
    shutil.copy2(src, dst)
    return dst


def show_figure(src: Path, height: int = 560):
    copied = copy_figure(src)
    if copied is None:
        display(Markdown(f"_Figure non disponible : `{src.name}`._"))
        return
    rel = Path("..") / "exports" / "figures" / "notebook_03_synthese" / copied.name
    display(IFrame(src=rel.as_posix(), width="100%", height=height))


portfolio_summary = read_sheet(NB01_XLSX, "Portfolio_Summary")
portfolio_composition = read_sheet(NB01_XLSX, "Portfolio_Composition")
optimisable_pocket = read_sheet(NB01_XLSX, "Optimisable_Pocket")
non_optimisable = read_sheet(NB01_XLSX, "Non_Optimisable")
risk_distribution = read_sheet(NB01_XLSX, "Portfolio_Risk_Distribution")
extreme_returns = read_sheet(
    NB01_XLSX,
    "Extreme_Historical_Returns",
    fallback=read_csv(DATA_EXPORTS / "extreme_historical_returns_table.csv"),
)
expected_scenarios_01 = read_sheet(
    NB01_XLSX,
    "Expected_Return_Scenarios",
    fallback=read_csv(DATA_EXPORTS / "expected_returns_scenarios.csv"),
)
apt_gap = read_sheet(NB01_XLSX, "APT_vs_Historical_Gap")
apt_factor_diag = read_sheet(
    NB01_XLSX,
    "APT_Factor_Diagnostics_Short",
    fallback=read_csv(DATA_EXPORTS / "apt_factor_diagnostics.csv"),
)

current_portfolio = read_sheet(NB02_XLSX, "02_Current_Portfolio")
return_scenarios = read_sheet(NB02_XLSX, "03_Return_Scenarios")
results_central = read_sheet(NB02_XLSX, "07_Results_APT_Central")
weights_model = read_sheet(NB02_XLSX, "10_Weights_By_Model")
weights_class = read_sheet(NB02_XLSX, "11_Weights_By_Class")
backtest = read_sheet(NB02_XLSX, "14_Backtest_Worst_10_Days")
stress_tests = read_sheet(NB02_XLSX, "15_Stress_Tests")
final_scoring = read_sheet(NB02_XLSX, "18_Final_Scoring")
recommended_portfolio = read_sheet(NB02_XLSX, "19_Recommended_Portfolio")
add10_hyp = read_sheet(NB02_XLSX, "21_Add10MD_Hypotheses")
add10_results = read_sheet(NB02_XLSX, "22_Add10MD_Results_By_Scenario")
add10_asset = read_sheet(NB02_XLSX, "23_Add10MD_Allocation_By_Asset")
add10_class = read_sheet(NB02_XLSX, "24_Add10MD_Allocation_By_Class")
add10_gap = read_sheet(NB02_XLSX, "27_Add10MD_Target_Gap")
add10_reco = read_sheet(NB02_XLSX, "29_Add10MD_Final_Recommendation")
capital_social = read_sheet(NB02_XLSX, "05_Capital_Social_Check")

SCENARIO_ORDER = ["APT_Prudent", "APT_Central", "APT_Optimistic", "Historical_Raw"]

with pd.ExcelWriter(SYNTH_XLSX, engine="openpyxl") as writer:
    sheets = {
        "01_Portefeuille": portfolio_summary,
        "02_Risque_Actuel": risk_distribution,
        "03_Extreme_Returns": extreme_returns,
        "04_Scenarios": return_scenarios,
        "05_Resultats_Central": results_central,
        "06_Scoring_Final": final_scoring,
        "07_Add10MD_Resultats": add10_results,
        "08_Add10MD_Reco": add10_reco,
        "09_Limites": capital_social,
    }
    for sheet, df in sheets.items():
        if df is not None and not df.empty:
            df.to_excel(writer, sheet_name=sheet[:31], index=False)

display(Markdown(f"Exports chargés. Synthèse Excel créée : `{SYNTH_XLSX.relative_to(PROJECT_ROOT).as_posix()}`."))

Exports chargés. Synthèse Excel créée : `exports/notebook_03_synthese_resultats.xlsx`.

## 2. Rappel du périmètre

        La démarche se concentre sur la poche réellement optimisable : titres de l’État, obligations corporate et actions cotées. Les poches figées ou non modélisées restent hors décision d’allocation, ce qui permet de ne pas attribuer au modèle une liberté de gestion que Maghrebia n’a pas nécessairement dans la pratique.

In [2]:
display_table(portfolio_summary, n=5)
display_table(current_portfolio, ["Asset", "Asset_Name", "Asset_Class", "Current_Value", "Current_Weight"], n=12)

,total_portfolio_value,optimisable_value,optimisable_weight,non_optimisable_value,non_optimisable_weight,reconciliation_gap_vs_expected,quality_flag
0,509032582.040245,330864900.909490,0.649988,178167681.130755,0.350012,0.000064,OK


,Asset,Asset_Name,Asset_Class,Current_Value,Current_Weight
0,BTA_7_4_02_2030,"BTA 7,4% 02/2030",Titres de l'État,32460797.551930,9.81%
1,BTA_7_5_12_2028,"BTA 7,5% 12/2028",Titres de l'État,22596740.738000,6.83%
2,EMPRUNT_NATIONAL_2021_2EME_TRANCHE,EMPRUNT NATIONAL 2021 2ème tranche,Titres de l'État,28923357.960000,8.74%
3,EMPRUNT_NATIONAL_2022_3EME_TRANCHE_10_ANS,EMPRUNT NATIONAL 2022 3ème tranche 10 ANS,Titres de l'État,14508276.320000,4.38%
4,EMPRUNT_NATIONAL_2023_4EME_TRANCHE_CATEGORIE_C,EMPRUNT NATIONAL 2023 4ème tranche Catégorie C,Titres de l'État,16438610.880000,4.97%
5,TN0007310568,E.O HL 2020-03,Obligations corporate,3298941.187500,1.00%
6,TN0003100872,EO BNA SUB 2021-1,Obligations corporate,6297656.389060,1.90%
7,TNHFN9X6ARP3,EO ADVANS 2022-1,Obligations corporate,9483725.240000,2.87%
8,TN8DSPQCBC06,EO ATL 2022-1,Obligations corporate,2790775.556000,0.84%
9,TNL7VQZVHR54,EO HL 2023-1,Obligations corporate,137893740.600000,41.68%


**Interprétation.** La distinction entre poche optimisable et poche non optimisable est structurante : l’analyse ne cherche pas à transformer tout le bilan financier, mais à améliorer la partie du portefeuille sur laquelle une décision d’allocation est réaliste. Pour Maghrebia, cette lecture est plus défendable qu’une optimisation théorique du portefeuille complet.

## 3. Diagnostic du portefeuille actuel

        Cette partie reprend les éléments de diagnostic du notebook 01. Les graphiques décrivent le comportement observé du portefeuille et ne doivent pas être lus comme une prévision. Ils servent à comprendre le point de départ avant optimisation.

### Composition de la poche optimisable

In [3]:
show_figure(FIG_NB01 / 'fig_05_composition_poche_optimisable.html', height=520)

**Lecture du graphique.** Le graphique montre comment la poche optimisable est répartie entre classes d’actifs et entre lignes individuelles.

        **Interprétation financière.** Cette composition donne le point de départ de l’optimisation. Si une classe domine déjà fortement la poche, le modèle doit améliorer le couple rendement-risque sans créer une concentration difficile à défendre.

        **Point de vigilance.** La diversification en poids ne suffit pas : une ligne peu pondérée peut contribuer fortement au risque si elle est volatile, et une obligation revalorisée par modèle peut afficher une volatilité plus lissée que son risque économique réel.

### Rendement cumulé 2025

In [4]:
show_figure(FIG_NB01 / 'fig_09_rendement_cumule_2025.html', height=520)

**Lecture du graphique.** La courbe présente la performance réalisée sur 2025 à partir des rendements périodiques construits dans le notebook 01.

        **Interprétation financière.** Elle permet de situer la dynamique historique du portefeuille avant toute hypothèse prospective. Pour Maghrebia, cette performance est utile pour expliquer le passé, mais elle ne doit pas être extrapolée mécaniquement.

        **Point de vigilance.** Une bonne performance sur une période courte peut venir de conditions de marché favorables ou d’opérations sur titres. Elle ne constitue pas, à elle seule, une hypothèse de rendement attendu.

### Distribution des rendements

In [5]:
show_figure(FIG_NB01 / 'fig_01_distribution_portefeuille_actuel.html', height=520)
display_table(risk_distribution, n=5)

,mean,volatility,annualized_mean_arithmetic,annualized_volatility,skewness,kurtosis_excess,Jarque_Bera_statistic,Jarque_Bera_pvalue,Shapiro_statistic,Shapiro_pvalue,Anderson_Darling_statistic,Anderson_Darling_5pct_critical,VaR_95,CVaR_95,max_drawdown,Historical_Cumulative_Return_2025,observations
0,0.000637,0.000826,0.160452,0.013111,0.405013,2.551565,69.467078,0.000000,0.958006,0.000001,2.307687,0.750000,0.05%,0.10%,-0.002807,16.94%,246


**Lecture du graphique.** La distribution résume la fréquence des rendements observés et met en évidence la dispersion autour du rendement moyen.

        **Interprétation financière.** Si la distribution est asymétrique ou présente des queues épaisses, la variance seule devient insuffisante pour piloter le risque. C’est précisément la raison pour laquelle le notebook 02 complète Markowitz par la CVaR.

        **Point de vigilance.** Les séries obligataires peuvent être partiellement lissées par la valorisation modèle. Les métriques historiques sont donc utiles, mais doivent être interprétées avec prudence.

### Drawdown et pertes extrêmes

In [6]:
show_figure(FIG_NB01 / 'fig_02_drawdown_portefeuille_actuel.html', height=500)
show_figure(FIG_NB01 / 'fig_03_var_cvar_95.html', height=500)

**Lecture du graphique.** Le drawdown mesure la perte depuis un point haut, tandis que la VaR et la CVaR résument les pertes défavorables observées.

        **Interprétation financière.** Pour une compagnie d’assurances, la perte extrême est plus importante qu’une simple volatilité moyenne. La CVaR est donc un indicateur central dans la recommandation, car elle regarde la sévérité des mauvaises observations plutôt que leur seule fréquence.

        **Point de vigilance.** Ces mesures restent estimées sur l’échantillon disponible. Elles aident à classer les portefeuilles, mais ne remplacent pas une politique complète de stress testing.

## 4. Rendements historiques élevés

        Le notebook 01 isole les actifs dont le rendement historique 2025 est particulièrement élevé. Ces rendements ne sont pas supprimés : ils sont documentés, puis utilisés uniquement dans le scénario comparatif `Historical_Raw`.

In [7]:
display_table(
    extreme_returns,
    [
        "Asset", "Asset_Class", "Historical_Cumulative_Return_2025",
        "Historical_Annualized_Arithmetic_Return", "Expected_Return_APT_Central",
        "Gap_Cumulative_vs_APT", "Data_Check_Status", "Explanation",
        "Methodological_Decision",
    ],
    n=20,
)

,Asset,Asset_Class,Historical_Cumulative_Return_2025,Historical_Annualized_Arithmetic_Return,Expected_Return_APT_Central,Gap_Cumulative_vs_APT,Data_Check_Status,Explanation,Methodological_Decision
0,PGH,Actions cotées,116.72%,81.34%,10.57%,106.15%,DESCRIPTIVE_HIGH_HISTORICAL_RETURN,PGH présente une forte revalorisation du titre sur 2025 et des fondamentaux favorables ; le rendement historique reste descriptif.,Conserver l'actif ; ne pas extrapoler automatiquement dans APT_Central.
1,DELICE HOLDING,Actions cotées,61.95%,52.37%,14.32%,47.63%,CORPORATE_ACTION_ADJUSTED,Délice Holding : split nominal 10 DT vers 5 DT au 01/08/2025 ; vérifier ou documenter l'ajustement corporate action.,Conserver Historical_Raw comme scénario comparatif ; utiliser APT_Central comme référence prospective.
2,ATTIJARI BANK,Actions cotées,55.45%,47.72%,13.90%,41.55%,CORPORATE_ACTION_ADJUSTED,Attijari Bank : augmentation de capital par incorporation de réserves et actions gratuites en 2025 ; l'ajustement doit être documenté.,Conserver Historical_Raw comme scénario comparatif ; ne pas extrapoler mécaniquement ce rendement.


**Interprétation.** Les cas comme PGH, Délice Holding ou Attijari Bank doivent être lus comme des performances observées, souvent liées à une forte revalorisation, à un contexte de marché favorable ou à des opérations sur titres. L’analyse ne les écarte pas, mais elle évite de les transformer directement en rendement espéré.

        Le rendement historique mesure ce qui s’est produit. Le rendement APT représente une hypothèse prospective plus prudente. `Historical_Raw` reste donc un scénario de comparaison, tandis que `APT_Central` demeure la référence principale de l’optimisation.

## 5. Scénarios de rendements attendus

        Les scénarios APT permettent de ne pas dépendre uniquement de l’historique 2025, qui peut contenir des performances exceptionnelles difficilement extrapolables. Ils structurent une lecture prudent, central et optimiste, complétée par `Historical_Raw` pour mesurer l’écart avec la performance brute observée.

In [8]:
display_table(
    return_scenarios,
    ["Asset", "Asset_Name", "Asset_Class", "APT_Prudent", "APT_Central", "APT_Optimistic", "Historical_Raw"],
    n=14,
)
show_figure(FIG_NB02 / '02_rendements_esperes_par_scenario.html', height=560)
show_figure(FIG_NB01 / 'fig_14_ecart_apt_central_vs_cumule_2025.html', height=520)

,Asset,Asset_Name,Asset_Class,APT_Prudent,APT_Central,APT_Optimistic,Historical_Raw
0,BTA_7_4_02_2030,"BTA 7,4% 02/2030",Titres de l'État,0.078372,0.100974,0.103586,0.111235
1,BTA_7_5_12_2028,"BTA 7,5% 12/2028",Titres de l'État,0.079532,0.100101,0.102966,0.108426
2,EMPRUNT_NATIONAL_2021_2EME_TRANCHE,EMPRUNT NATIONAL 2021 2ème tranche,Titres de l'État,0.092617,0.099397,0.102322,0.098383
3,EMPRUNT_NATIONAL_2022_3EME_TRANCHE_10_ANS,EMPRUNT NATIONAL 2022 3ème tranche 10 ANS,Titres de l'État,0.082846,0.100644,0.103305,0.107073
4,EMPRUNT_NATIONAL_2023_4EME_TRANCHE_CATEGORIE_C,EMPRUNT NATIONAL 2023 4ème tranche Catégorie C,Titres de l'État,0.083277,0.101508,0.103981,0.110124
5,TN0007310568,E.O HL 2020-03,Obligations corporate,0.103641,0.109896,0.117791,0.097950
6,TN0003100872,EO BNA SUB 2021-1,Obligations corporate,0.103205,0.108020,0.115892,0.096683
7,TNHFN9X6ARP3,EO ADVANS 2022-1,Obligations corporate,0.102621,0.112136,0.119955,0.107498
8,TN8DSPQCBC06,EO ATL 2022-1,Obligations corporate,0.098601,0.111068,0.119045,0.105398
9,TNL7VQZVHR54,EO HL 2023-1,Obligations corporate,0.096414,0.111433,0.119351,0.108759


**Interprétation.** Le scénario central donne une base de décision plus stable que le seul rendement historique. Le scénario prudent teste la sensibilité à un environnement moins favorable, tandis que le scénario optimiste mesure l’impact d’hypothèses plus porteuses mais encadrées. `Historical_Raw` est utile pour la discussion, notamment lorsque l’historique 2025 paraît très favorable, mais il n’est pas la base de recommandation.

## 6. Résultats d’optimisation

        Le notebook 02 compare plusieurs rôles de modèles : Markowitz comme benchmark académique, Max Return comme borne agressive sous contraintes, Mean-CVaR comme modèle institutionnel centré sur le risque extrême, Risk Parity pour la diversification du risque, Robust CVaR Conservative Proxy pour une lecture prudente, et Monte Carlo comme exploration de l’espace faisable.

In [9]:
display_table(
    results_central,
    [
        "Scenario", "Model", "Role", "Expected_Return", "Volatility", "VaR_95",
        "CVaR_95", "Max_Weight", "HHI", "Turnover", "Regulatory_Status", "Comment",
    ],
    n=15,
)
display_table(
    final_scoring,
    ["Scenario", "Model", "Role", "Final_Score", "Recommended", "Regulatory_Status", "Comment"],
    n=20,
    percent=False,
)
show_figure(FIG_NB02 / '03_performance_modeles_par_scenario.html', height=560)

,Scenario,Model,Role,Expected_Return,Volatility,VaR_95,CVaR_95,Max_Weight,HHI,Turnover,Regulatory_Status,Comment
0,APT_Central,Current_Portfolio,Portefeuille actuel,10.68%,1.55%,0.05%,0.10%,41.68%,0.206067,0.000000,NON_TESTABLE_DATA_MISSING,
1,APT_Central,Markowitz_Max_Return,Benchmark académique,11.46%,3.50%,0.17%,0.24%,30.00%,0.132520,0.300000,NON_TESTABLE_DATA_MISSING,Maximum Return = borne supérieure agressive.
2,APT_Central,Markowitz_Mean_Variance,Benchmark académique,11.43%,2.93%,0.09%,0.18%,30.00%,0.130323,0.300000,NON_TESTABLE_DATA_MISSING,
3,APT_Central,Mean_CVaR,Modèle institutionnel principal,10.91%,1.37%,0.02%,0.05%,30.00%,0.133994,0.300000,NON_TESTABLE_DATA_MISSING,
4,APT_Central,Risk_Parity,Diversification du risque,10.74%,1.10%,0.04%,0.09%,30.00%,0.139245,0.300000,NON_TESTABLE_DATA_MISSING,
5,APT_Central,Robust_CVaR_Conservative_Proxy,Test de robustesse,10.69%,0.48%,0.00%,0.00%,30.00%,0.138036,0.431391,NON_TESTABLE_DATA_MISSING,
6,APT_Central,Monte_Carlo_Max_Return,Exploration Monte Carlo,10.91%,1.95%,0.06%,0.13%,29.77%,0.124791,0.278812,NON_TESTABLE_DATA_MISSING,
7,APT_Central,Monte_Carlo_Min_Volatility,Exploration Monte Carlo,10.71%,1.30%,0.05%,0.09%,29.81%,0.133459,0.287189,NON_TESTABLE_DATA_MISSING,
8,APT_Central,Monte_Carlo_Min_CVaR,Exploration Monte Carlo,10.70%,1.35%,0.04%,0.08%,29.33%,0.131720,0.289526,NON_TESTABLE_DATA_MISSING,
9,APT_Central,Monte_Carlo_Best_Scoring,Exploration Monte Carlo,10.85%,1.57%,0.05%,0.10%,29.99%,0.128092,0.276071,NON_TESTABLE_DATA_MISSING,


,Scenario,Model,Role,Recommended,Regulatory_Status,Comment
0,APT_Prudent,Robust_CVaR_Conservative_Proxy,Test de robustesse,False,NON_TESTABLE_DATA_MISSING,
1,APT_Prudent,Mean_CVaR,Modèle institutionnel principal,False,NON_TESTABLE_DATA_MISSING,
2,APT_Prudent,Markowitz_Mean_Variance,Benchmark académique,False,NON_TESTABLE_DATA_MISSING,
3,APT_Prudent,Risk_Parity,Diversification du risque,False,NON_TESTABLE_DATA_MISSING,
4,APT_Prudent,Current_Portfolio,Portefeuille actuel,False,NON_TESTABLE_DATA_MISSING,
5,APT_Prudent,Markowitz_Max_Return,Benchmark académique,False,NON_TESTABLE_DATA_MISSING,Maximum Return = borne supérieure agressive.
6,APT_Prudent,Monte_Carlo_Max_Return,Exploration Monte Carlo,False,NON_TESTABLE_DATA_MISSING,
7,APT_Prudent,Monte_Carlo_Min_Volatility,Exploration Monte Carlo,False,NON_TESTABLE_DATA_MISSING,
8,APT_Prudent,Monte_Carlo_Min_CVaR,Exploration Monte Carlo,False,NON_TESTABLE_DATA_MISSING,
9,APT_Prudent,Monte_Carlo_Best_Scoring,Exploration Monte Carlo,False,NON_TESTABLE_DATA_MISSING,


**Interprétation.** Le classement ne doit pas être lu uniquement par le rendement. Pour Maghrebia, un portefeuille plus équilibré peut être plus défendable qu’un portefeuille très agressif si son risque extrême, sa concentration ou sa rotation sont mieux maîtrisés. Le scoring multicritère formalise cette lecture institutionnelle.

### Poids et lecture des allocations

In [10]:
show_figure(FIG_NB02 / '04_poids_par_classe_modeles.html', height=560)
show_figure(FIG_NB02 / '05_poids_par_actif_apt_central.html', height=560)

**Interprétation.** Les poids par classe donnent une lecture de gouvernance : ils indiquent si le modèle déplace le portefeuille vers les actions, les obligations corporate ou les titres de l’État. Les poids par actif complètent cette lecture en montrant les concentrations concrètes, ce qui est indispensable pour une décision opérationnelle.

## 7. Frontière efficiente et comparaison des modèles

        La frontière efficiente synthétise le compromis rendement-risque. Elle doit être lue avec les autres métriques : la volatilité ne capture pas tout le risque d’une compagnie d’assurances, en particulier le risque de queue mesuré par la CVaR.

In [11]:
show_figure(FIG_NB02 / '06_frontiere_efficiente_propre.html', height=620)

**Interprétation.** La frontière montre les allocations efficientes selon le couple rendement-risque. Les portefeuilles retenus doivent être lus non seulement selon leur rendement, mais aussi selon la CVaR, la concentration, le turnover et les contraintes. Une solution située correctement sur la frontière mais trop concentrée peut rester moins acceptable en gestion assurantielle.

## 8. Backtesting et stress tests

        Ces tests ne prédisent pas l’avenir. Ils permettent de vérifier si l’allocation proposée reste cohérente dans des conditions défavorables observées ou paramétrées.

In [12]:
display_table(backtest, n=12)
show_figure(FIG_NB02 / '07_backtest_pertes_moyennes_pires_seances.html', height=520)
display_table(stress_tests, n=14)
show_figure(FIG_NB02 / '08_stress_testing_impact.html', height=560)

,Scenario,Date,Worst_Day_Rank,Current_Portfolio_Return,Markowitz_Mean_Variance_Return,Mean_CVaR_Return,Risk_Parity_Return,Robust_CVaR_Conservative_Proxy_Return
0,APT_Prudent,2025-06-03 00:00:00,1,-0.28%,-0.14%,-0.10%,-0.19%,-0.02%
1,APT_Prudent,2025-05-20 00:00:00,2,-0.16%,-0.54%,-0.16%,-0.17%,0.01%
2,APT_Prudent,2025-01-03 00:00:00,3,-0.13%,-0.15%,-0.03%,-0.15%,-0.03%
3,APT_Prudent,2025-11-14 00:00:00,4,-0.11%,-0.06%,-0.03%,-0.08%,0.02%
4,APT_Prudent,2025-06-13 00:00:00,5,-0.10%,-0.10%,-0.06%,-0.10%,-0.01%
5,APT_Prudent,2025-01-30 00:00:00,6,-0.09%,-0.05%,-0.03%,-0.04%,0.01%
6,APT_Prudent,2025-03-07 00:00:00,7,-0.09%,-0.18%,-0.09%,-0.09%,0.01%
7,APT_Prudent,2025-06-17 00:00:00,8,-0.08%,0.03%,-0.03%,-0.02%,0.01%
8,APT_Prudent,2025-03-11 00:00:00,9,-0.07%,-0.06%,-0.02%,-0.06%,0.01%
9,APT_Prudent,2025-01-21 00:00:00,10,-0.06%,-0.06%,-0.05%,0.02%,0.02%


,Scenario,Stress_Scenario,Model,Equity_Impact,Bond_Impact,Estimated_Return_Impact,Estimated_Loss,Duration_Status,Comment
0,APT_Prudent,Equity_Shock_Mild,Current_Portfolio,-0.012178,0.000000,-1.22%,0.012178,DURATION_PROXY_RESIDUAL_MATURITY,"Proxy basé sur la maturité résiduelle, à interpréter avec prudence."
1,APT_Prudent,Equity_Shock_Severe,Current_Portfolio,-0.030445,0.000000,-3.04%,0.030445,DURATION_PROXY_RESIDUAL_MATURITY,"Proxy basé sur la maturité résiduelle, à interpréter avec prudence."
2,APT_Prudent,Rate_Shock_100bps,Current_Portfolio,0.000000,-0.025442,-2.54%,0.025442,DURATION_PROXY_RESIDUAL_MATURITY,"Proxy basé sur la maturité résiduelle, à interpréter avec prudence."
3,APT_Prudent,Rate_Shock_200bps,Current_Portfolio,0.000000,-0.050884,-5.09%,0.050884,DURATION_PROXY_RESIDUAL_MATURITY,"Proxy basé sur la maturité résiduelle, à interpréter avec prudence."
4,APT_Prudent,Credit_Spread_Shock_100bps,Current_Portfolio,0.000000,-0.011917,-1.19%,0.011917,DURATION_PROXY_RESIDUAL_MATURITY,"Proxy basé sur la maturité résiduelle, à interpréter avec prudence."
5,APT_Prudent,Credit_Spread_Shock_200bps,Current_Portfolio,0.000000,-0.023835,-2.38%,0.023835,DURATION_PROXY_RESIDUAL_MATURITY,"Proxy basé sur la maturité résiduelle, à interpréter avec prudence."
6,APT_Prudent,Combined_Adverse,Current_Portfolio,-0.024356,-0.056039,-8.04%,0.080395,DURATION_PROXY_RESIDUAL_MATURITY,"Proxy basé sur la maturité résiduelle, à interpréter avec prudence."
7,APT_Prudent,Geopolitical_Regional_Stress,Current_Portfolio,-0.030445,-0.049277,-7.97%,0.079722,DURATION_PROXY_RESIDUAL_MATURITY,"Proxy basé sur la maturité résiduelle, à interpréter avec prudence."
8,APT_Prudent,Optimistic_Normalization,Current_Portfolio,0.012178,0.018680,3.09%,-0.030858,DURATION_PROXY_RESIDUAL_MATURITY,"Proxy basé sur la maturité résiduelle, à interpréter avec prudence."
9,APT_Prudent,Equity_Shock_Mild,Markowitz_Max_Return,-0.023855,0.000000,-2.39%,0.023855,DURATION_PROXY_RESIDUAL_MATURITY,"Proxy basé sur la maturité résiduelle, à interpréter avec prudence."


**Interprétation.** Le backtesting sur les pires séances donne une lecture concrète du comportement en périodes défavorables déjà observées. Les stress tests paramétriques complètent cette lecture en simulant des chocs actions, taux et spreads. Pour Maghrebia, l’intérêt est de ne pas choisir une allocation uniquement parce qu’elle améliore le rendement attendu, mais parce qu’elle reste compréhensible sous stress.

## 9. Allocation optimisée de 10 MD

        La section finale du notebook 02 optimise l’allocation d’une enveloppe additionnelle de 10 MD dans la poche optimisable. Les positions existantes ne sont pas vendues ; les métriques sont recalculées sur le portefeuille final après ajout.

        L’objectif demandé est `ROE = TSR + 4 %`. Dans l’analyse, il est approché par une cible de rendement financier attendu, car le modèle ne calcule pas le résultat net comptable ni les fonds propres.

In [13]:
display_table(add10_hyp, n=10, percent=False)
display_table(
    add10_results,
    [
        "Scenario", "Variant", "Model", "R_Additional", "R_OPT_FINAL",
        "R_TOTAL_MODELLED_PROXY_FINAL", "Gap_Total_Modelled_To_ROE_Target",
        "Max_New_Money_Weight", "Effective_Number_New_Money", "Concentration_Status",
        "Add10MD_Recommended", "Comment",
    ],
    n=20,
)

,Indicateur,Valeur,Commentaire
0,V_TOTAL_CURRENT,509032582.040245,Portefeuille total actuel
1,V_OPT_CURRENT,330864900.909490,Poche optimisable actuelle
2,V_FIXED_CURRENT,178167681.130755,Poche non optimisable figée
3,ADDITIONAL_BUDGET,10000000.000000,Investi uniquement dans la poche optimisable
4,V_OPT_FINAL,340864900.909490,V_OPT_CURRENT + ADDITIONAL_BUDGET
5,V_TOTAL_FINAL,519032582.040245,V_TOTAL_CURRENT + ADDITIONAL_BUDGET
6,ROE_Target_TSR_Plus_4,0.116408,"Objectif demandé : ROE = TSR + 4 %, approximé par une cible de rendement financier attendu"


,Scenario,Model,R_OPT_FINAL,R_TOTAL_MODELLED_PROXY_FINAL,Gap_Total_Modelled_To_ROE_Target,Max_New_Money_Weight,Effective_Number_New_Money,Concentration_Status
0,APT_Prudent,Add10MD_Target_Seeking_Markowitz_Max_Return,9.23%,6.06%,-5.58%,100.00%,1.000000,HIGH_CONCENTRATION
1,APT_Prudent,Add10MD_Target_Seeking_Markowitz_Mean_Variance,9.23%,6.06%,-5.58%,100.00%,1.000000,HIGH_CONCENTRATION
2,APT_Prudent,Add10MD_Target_Seeking_Mean_CVaR,9.23%,6.06%,-5.58%,100.00%,1.000000,HIGH_CONCENTRATION
3,APT_Prudent,Add10MD_Target_Seeking_Risk_Parity,9.20%,6.04%,-5.60%,64.90%,1.836924,HIGH_CONCENTRATION
4,APT_Prudent,Add10MD_Target_Seeking_Robust_CVaR_Conservative_Proxy,9.22%,6.06%,-5.58%,86.50%,1.304603,HIGH_CONCENTRATION
5,APT_Prudent,Add10MD_Target_Seeking_Monte_Carlo_Best_Scoring,9.12%,5.99%,-5.65%,10.45%,15.139402,DIVERSIFIED
6,APT_Prudent,Add10MD_Diversified_Markowitz_Max_Return,9.18%,6.03%,-5.61%,25.00%,4.000000,DIVERSIFIED
7,APT_Prudent,Add10MD_Diversified_Markowitz_Mean_Variance,9.18%,6.03%,-5.61%,25.00%,4.000000,DIVERSIFIED
8,APT_Prudent,Add10MD_Diversified_Mean_CVaR,9.15%,6.01%,-5.63%,25.00%,4.000000,DIVERSIFIED
9,APT_Prudent,Add10MD_Diversified_Risk_Parity,8.94%,5.87%,-5.77%,25.00%,4.000000,DIVERSIFIED


### Allocation de l’enveloppe par classe et par actif

In [14]:
show_figure(FIG_ADD10 / '02_allocation_10md_classe_scenario.html', height=560)
show_figure(FIG_ADD10 / '03_top_actifs_add10md_scenario.html', height=560)

**Interprétation.** La solution `Target_Seeking` maximise le rapprochement vers l’objectif, mais peut concentrer le nouveau budget. La variante `Diversified` réduit cette concentration avec une perte de rendement marginale ; elle est donc plus prudente opérationnellement lorsque l’écart de rendement reste faible.

### Impact sur la poche optimisable et le portefeuille total modélisé

In [15]:
show_figure(FIG_ADD10 / '04_rendement_final_poche_optimisable.html', height=520)
show_figure(FIG_ADD10 / '05_rendement_total_modelise_final.html', height=520)
show_figure(FIG_ADD10 / '06_gap_cible_scenario_modele.html', height=560)

**Interprétation.** L’ajout de 10 MD améliore le rendement attendu de la poche optimisable. Son effet sur le portefeuille total reste plus limité, car l’enveloppe additionnelle représente une fraction du portefeuille global. Cette distinction est importante : une allocation pertinente sur la poche optimisable peut ne pas suffire, seule, à combler l’écart à la cible globale.

## 10. Recommandation finale

        La recommandation principale reste rattachée au scénario `APT_Central`. `Historical_Raw` sert à éclairer l’écart avec l’historique 2025, mais ne constitue pas une base prospective de décision.

In [16]:
display_table(recommended_portfolio, n=12, percent=False)
display_table(add10_reco, n=20, percent=False)
show_figure(FIG_ADD10 / '07_comparaison_modeles_add10md_apt_central.html', height=560)

,Scenario,Model,Role,Expected_Return,Volatility,VaR_95,CVaR_95,Max_Weight,HHI,Turnover,Return_to_RiskPenalty,Regulatory_Status,Capital_Social_Status,Recommended,Comment
0,APT_Prudent,Current_Portfolio,Portefeuille actuel,0.091231,0.015454,0.000499,0.001023,0.416768,0.206067,0.000000,0.717851,NON_TESTABLE_DATA_MISSING,NON_TESTABLE_DATA_MISSING,False,
1,APT_Prudent,Markowitz_Max_Return,Benchmark académique,0.096057,0.033969,0.001346,0.002442,0.300000,0.136893,0.300000,1.618885,NON_TESTABLE_DATA_MISSING,NON_TESTABLE_DATA_MISSING,False,Maximum Return = borne supérieure agressive.
2,APT_Prudent,Markowitz_Mean_Variance,Benchmark académique,0.095495,0.019391,0.000472,0.001284,0.300000,0.137753,0.300000,1.617547,NON_TESTABLE_DATA_MISSING,NON_TESTABLE_DATA_MISSING,False,
3,APT_Prudent,Mean_CVaR,Modèle institutionnel principal,0.093755,0.013374,0.000254,0.000562,0.300000,0.142644,0.300000,1.483337,NON_TESTABLE_DATA_MISSING,NON_TESTABLE_DATA_MISSING,False,
4,APT_Prudent,Risk_Parity,Diversification du risque,0.085972,0.010959,0.000400,0.000881,0.300000,0.139245,0.300000,1.429871,NON_TESTABLE_DATA_MISSING,NON_TESTABLE_DATA_MISSING,False,
5,APT_Prudent,Robust_CVaR_Conservative_Proxy,Test de robustesse,0.090251,0.004841,0.000000,0.000039,0.300000,0.138036,0.431390,1.554048,NON_TESTABLE_DATA_MISSING,NON_TESTABLE_DATA_MISSING,False,
6,APT_Prudent,Monte_Carlo_Max_Return,Exploration Monte Carlo,0.090725,0.018886,0.000591,0.001245,0.292109,0.126679,0.261278,1.893100,NON_TESTABLE_DATA_MISSING,NON_TESTABLE_DATA_MISSING,False,
7,APT_Prudent,Monte_Carlo_Min_Volatility,Exploration Monte Carlo,0.088935,0.012956,0.000457,0.000904,0.298149,0.133459,0.287189,1.635965,NON_TESTABLE_DATA_MISSING,NON_TESTABLE_DATA_MISSING,False,
8,APT_Prudent,Monte_Carlo_Min_CVaR,Exploration Monte Carlo,0.087210,0.013548,0.000424,0.000829,0.293269,0.131720,0.289526,1.659609,NON_TESTABLE_DATA_MISSING,NON_TESTABLE_DATA_MISSING,False,
9,APT_Prudent,Monte_Carlo_Best_Scoring,Exploration Monte Carlo,0.090208,0.015265,0.000601,0.001074,0.297365,0.128323,0.253068,1.826181,NON_TESTABLE_DATA_MISSING,NON_TESTABLE_DATA_MISSING,False,


,Scenario,Add10MD_Variant,Model_Family,Model,Model_Display,R_ADDITIONAL,R_OPT_FINAL,R_TOTAL_MODELLED_PROXY_FINAL,Gap_Opt_To_ROE_Target,Gap_Total_Modelled_To_ROE_Target,Volatility_Final,VaR_95_Final,CVaR_95_Final,HHI_Final,New_Money_HHI,Effective_Number_New_Money,Max_New_Money_Weight,Max_Weight_Final,Concentration_Status,Regulatory_Status,Capital_Social_Status,Weights_JSON,Target_Status,Final_Status,Target_Achievement_Component,Regulatory_Component,Diversification_Component,Add10MD_Score,Concentration_Comment,Recommendation_Role
0,APT_Central,Add10MD_Target_Seeking,Markowitz_Mean_Variance,Add10MD_Target_Seeking_Markowitz_Mean_Variance,Target_Seeking | Markowitz_Mean_Variance,0.160979,0.108414,0.071199,-0.007993,-0.045209,0.017392,0.000511,0.000979,0.195022,1.000000,1,1.000000,0.265675,HIGH_CONCENTRATION,PASSED,NON_TESTABLE_DATA_MISSING,"{""BTA_7_4_02_2030"":0.0,""BTA_7_5_12_2028"":0.0,""EMPRUNT_NATIONAL_2021_2EME_TRANCHE"":0.0,""EMPRUNT_NATIONAL_2022_3EME_TRANCHE_10_ANS"":0.0,""EMPRUNT_NATIONAL_2023_4EME_TRANCHE_CATEGORIE_C"":0.0,""TN0007310568"":0.0,""TN0003100872"":0.0,""TNHFN9X6ARP3"":0.0,""TN8DSPQCBC06"":0.0,""TNL7VQZVHR54"":0.0,""TNYGOCQ2WZX6"":0.0,""TNDE9EH7SA12"":0.0,""EO_ATL_2025_2"":0.0,""AMEN_BANK"":0.0,""ATTIJARI_BANK"":0.0,""BIAT"":0.0,""BT"":1.0,""CITY_CARS"":0.0,""DELICE_HOLDING"":0.0,""ENNAKL"":0.0,""PGH"":0.0}",TARGET_NOT_REACHED,ANALYSIS_VALID_TARGET_NOT_REACHED,0.611636,1,0.250000,0.763587,Allocation agressive à comparer à la variante diversifiée,Target_Seeking_Reference
1,APT_Central,Add10MD_Diversified,Markowitz_Mean_Variance,Add10MD_Diversified_Markowitz_Mean_Variance,Diversified | Markowitz_Mean_Variance,0.144813,0.107940,0.070888,-0.008468,-0.045520,0.016640,0.000611,0.001090,0.194678,0.250000,4,0.250000,0.265675,DIVERSIFIED,PASSED,NON_TESTABLE_DATA_MISSING,"{""BTA_7_4_02_2030"":0.0,""BTA_7_5_12_2028"":0.0,""EMPRUNT_NATIONAL_2021_2EME_TRANCHE"":0.0,""EMPRUNT_NATIONAL_2022_3EME_TRANCHE_10_ANS"":0.0,""EMPRUNT_NATIONAL_2023_4EME_TRANCHE_CATEGORIE_C"":0.0,""TN0007310568"":0.0,""TN0003100872"":0.0,""TNHFN9X6ARP3"":0.0,""TN8DSPQCBC06"":0.0,""TNL7VQZVHR54"":0.0,""TNYGOCQ2WZX6"":0.0,""TNDE9EH7SA12"":0.0,""EO_ATL_2025_2"":0.0,""AMEN_BANK"":0.0,""ATTIJARI_BANK"":0.25,""BIAT"":0.0,""BT"":0.25,""CITY_CARS"":0.25,""DELICE_HOLDING"":0.25,""ENNAKL"":0.0,""PGH"":0.0}",TARGET_NOT_REACHED,ANALYSIS_VALID_TARGET_NOT_REACHED,0.608960,1,1.000000,0.641062,Concentration suivie comme indicateur de prudence,Operational_Prudent_Recommendation


**Interprétation.** La lecture centrale privilégie une allocation issue du scoring sous `APT_Central`. Lorsque la variante diversifiée conserve un rendement proche de la solution orientée cible, elle offre une traduction plus prudente pour la gestion réelle : moins de concentration du nouveau budget, meilleure lisibilité et décision plus facile à défendre.

## 11. Limites

        Plusieurs limites restent à garder visibles. L’historique 2025 est court et peut contenir des performances exceptionnelles. La liquidité de certaines lignes, notamment obligataires, n’est pas entièrement modélisée. Les valorisations obligataires par modèle peuvent lisser une partie de la volatilité observée.

        La contrainte liée au capital social de l’émetteur dépend de la disponibilité des `Shares_Outstanding`. Lorsqu’elle manque, la contrainte est documentée mais non testable ; elle ne doit pas être considérée comme validée.

        Enfin, le rendement total utilisé dans la section 10 MD est un proxy modélisé : il sert à apprécier l’effet marginal de l’enveloppe additionnelle, sans prétendre remplacer un calcul complet du ROE comptable.

In [17]:
display_table(capital_social, n=12, percent=False)

,File,Sheets,Columns,Has_Capital_Social_Data,Decision,Table,Asset,Issuer,Capital_Social_Holding_Ratio,Status,Reason,Required_Data,Available_Data
0,data\raw\capital_social_reference_template.csv,CSV,"Issuer, Ticker, ISIN, Shares_Outstanding, Source, Reference_Date, Comment",0.000000,TEMPLATE_EMPTY_NOT_DATA,Source_Check,,,,,,,
1,,,,,,Capital_Check,BTA_7_4_02_2030,BTA_7_4_02_2030,,NON_TESTABLE_DATA_MISSING,Référence capital social absente.,Shares_Outstanding,Positions uniquement
2,,,,,,Capital_Check,BTA_7_5_12_2028,BTA_7_5_12_2028,,NON_TESTABLE_DATA_MISSING,Référence capital social absente.,Shares_Outstanding,Positions uniquement
3,,,,,,Capital_Check,EMPRUNT_NATIONAL_2021_2EME_TRANCHE,EMPRUNT_NATIONAL_2021_2EME_TRANCHE,,NON_TESTABLE_DATA_MISSING,Référence capital social absente.,Shares_Outstanding,Positions uniquement
4,,,,,,Capital_Check,EMPRUNT_NATIONAL_2022_3EME_TRANCHE_10_ANS,EMPRUNT_NATIONAL_2022_3EME_TRANCHE_10_ANS,,NON_TESTABLE_DATA_MISSING,Référence capital social absente.,Shares_Outstanding,Positions uniquement
5,,,,,,Capital_Check,EMPRUNT_NATIONAL_2023_4EME_TRANCHE_CATEGORIE_C,EMPRUNT_NATIONAL_2023_4EME_TRANCHE_CATEGORIE_C,,NON_TESTABLE_DATA_MISSING,Référence capital social absente.,Shares_Outstanding,Positions uniquement
6,,,,,,Capital_Check,TN0007310568,TN0007310568,,NON_TESTABLE_DATA_MISSING,Référence capital social absente.,Shares_Outstanding,Positions uniquement
7,,,,,,Capital_Check,TN0003100872,TN0003100872,,NON_TESTABLE_DATA_MISSING,Référence capital social absente.,Shares_Outstanding,Positions uniquement
8,,,,,,Capital_Check,TNHFN9X6ARP3,TNHFN9X6ARP3,,NON_TESTABLE_DATA_MISSING,Référence capital social absente.,Shares_Outstanding,Positions uniquement
9,,,,,,Capital_Check,TN8DSPQCBC06,TN8DSPQCBC06,,NON_TESTABLE_DATA_MISSING,Référence capital social absente.,Shares_Outstanding,Positions uniquement


## 12. Conclusion générale

        Le projet apporte une chaîne de décision reproductible : construction des rendements, séparation entre historique et rendement espéré, scénarios APT, covariance shrinkée, optimisation sous contraintes, stress tests et scoring multicritère. Cette démarche donne à Maghrebia une lecture plus structurée que la simple extrapolation de la performance 2025.

        Les résultats suggèrent que le scénario `APT_Central` doit rester la référence de décision. Les rendements historiques élevés sont utiles pour expliquer le passé, mais ils ne suffisent pas à fonder une allocation prospective. La recommandation s’appuie donc sur un compromis entre rendement attendu, risque extrême, concentration, contraintes testables et lisibilité opérationnelle.

        L’ajout de 10 MD améliore le rendement attendu, mais son impact global reste limité par l’effet de taille. La décision finale ne dépend donc pas uniquement du choix des actifs recevant les 10 MD ; elle dépend aussi de la structure globale du portefeuille et du niveau de risque que Maghrebia accepte de porter.